# Module 04 — Grader Engineering

Understand and extend the grading system.

## Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '../../src'))
from openenv_env.graders import (
    syntax_grader, output_grader, similarity_grader,
    explanation_grader, composite_grader, GRADER_REGISTRY
)
from openenv_env.models import Action
from openenv_env.tasks import TASK_REGISTRY

## Test Each Grader

In [ ]:
task = TASK_REGISTRY["task_syntax_001"]

correct = Action(
    fixed_code="def multiply(a, b):\n    result = a * b\n    return result\n",
    explanation="Added missing colon after function signature.",
    confidence=0.9
)
wrong = Action(fixed_code=task.buggy_code, explanation="", confidence=0.0)

for name, fn in GRADER_REGISTRY.items():
    c = fn(correct, task)
    w = fn(wrong, task)
    print(f"{name:15s}  correct={c:.3f}  wrong={w:.3f}")

## Visualise Reward Breakdown

In [ ]:
try:
    import matplotlib.pyplot as plt
    names = list(GRADER_REGISTRY.keys())[:-1]  # skip composite
    correct_scores = [GRADER_REGISTRY[n](correct, task) for n in names]
    wrong_scores   = [GRADER_REGISTRY[n](wrong, task)   for n in names]
    x = range(len(names))
    plt.figure(figsize=(8, 4))
    plt.bar([i - 0.2 for i in x], correct_scores, 0.4, label="correct fix", color="green")
    plt.bar([i + 0.2 for i in x], wrong_scores,   0.4, label="wrong fix",   color="red")
    plt.xticks(list(x), names)
    plt.ylabel("Score")
    plt.title("Grader scores: correct vs wrong fix")
    plt.legend()
    plt.tight_layout()
    plt.savefig("grader_comparison.png")
    print("Saved grader_comparison.png")
except ImportError:
    print("matplotlib not installed — skipping plot")

## Write a Custom Grader

In [ ]:
def line_count_grader(action: Action, task) -> float:
    """Reward fixes that are concise (fewer lines = more elegant)."""
    ref_lines   = len(task.fixed_code.strip().splitlines())
    agent_lines = len(action.fixed_code.strip().splitlines())
    if agent_lines == 0:
        return 0.0
    # Perfect score if same length; penalty for bloat
    ratio = ref_lines / agent_lines
    return round(min(ratio, 1.0), 4)

print("Custom grader score:", line_count_grader(correct, task))

## Register and Test

In [ ]:
GRADER_REGISTRY["line_count"] = line_count_grader

# Re-run composite (it does NOT include custom graders unless you update weights)
for name, fn in GRADER_REGISTRY.items():
    print(f"{name:15s}: {fn(correct, task):.3f}")

## Exercise
Write a grader that checks whether the agent's fix preserves the original function signature (name + parameters). Score 1.0 if preserved, 0.0 otherwise.